### 1. Análises de qualidade e MultiQC

In [ ]:
%%bash

fastqc \
    -t 4 \
    -o controle_qualidade/fastqc *.fastq.gz

multiqc \
    controle_qualidade/fastqc \
    -o controle_qualidade/multiqc

[Relatório MultiQC - before-trimming](../results/multiqc_before_trim.html)

### 2. Trimmagem - Remoção de adaptadores e filtragem de bases e reads

In [ ]:
%%bash

for R1 in /media/lgbio-nas1/kassia.miranda/lcWGS_macauba/*_R1_001.fastq.gz; do
    SAMPLE=$(basename "$R1" _R1_001.fastq.gz)
    R2="/media/lgbio-nas1/kassia.miranda/lcWGS_macauba/${SAMPLE}_R2_001.fastq.gz"
    fastp \
        -i "$R1" \
        -I "$R2" \
        -o "/media/lgbio-nas1/kassia.miranda/lcWGS_macauba/controle_qualidade/trimmagem/${SAMPLE}_R1_001.trim.fq.gz" \
        -O "/media/lgbio-nas1/kassia.miranda/lcWGS_macauba/controle_qualidade/trimmagem/${SAMPLE}_R2_001.trim.fq.gz" \
        --detect_adapter_for_pe \
        -j "${SAMPLE}.fastp.trim.json"
        -q 30 \
        -u 40 \
        -w 24
done

[Relatório MultiQC - pós-trimming](../results/multiqc_post_trim.html)

---

### 3. Indexando genoma de referência

In [ ]:
bwa-mem2 index genome.fasta

---

### 4. Alinhado os fastq à referência

In [ ]:
%%bash

#!/usr/bin/env bash

set -euo pipefail

# ---- configuração do projeto (não dá para obter do FASTQ) ----
GENOME="/media/lgbio-nas1/kassia.miranda/lcWGS_macauba/genome/macauba_genoma.fna"
THREADS=14
SORT_THREADS=4
CN="LGBio"
DT="2026-09-09"
LB_PROJETO="LcWGS_Macauba"
OUTDIR="bam_sorted"

mkdir -p "$OUTDIR"

# checa o índice do bwa-mem2 antes de começar
for EXT in 0123 amb ann bwt.2bit.64 pac; do
    [[ -f "${GENOME}.${EXT}" ]] || { echo "ERRO: índice ausente: ${GENOME}.${EXT} (rode: bwa-mem2 index $GENOME)" >&2; exit 1; }
done

for R1 in *_R1_001.trim.fq.gz; do
    R2="${R1/_R1_001/_R2_001}"
    [[ -f "$R2" ]] || { echo "AVISO: sem R2 para $R1, pulando" >&2; continue; }

    # AACANAMS_11-1_S69_L001
    BASE="${R1%_R1_001.trim.fq.gz}"
    # AACANAMS_11-1  (remove _S69_L001)
    SM="$(sed -E 's/_S[0-9]+_L[0-9]{3}$//' <<< "$BASE")"

    OUT="${OUTDIR}/${BASE}.sort.bam"
    TMP="${OUT}.tmp"

    # pula o que já foi feito
    if [[ -s "$OUT" && -s "${OUT}.bai" ]]; then
        echo "OK: $OUT já existe, pulando" >&2
        continue
    fi

    # primeira linha do FASTQ: @instrumento:run:flowcell:lane:tile:x:y 1:N:0:barcode
    HDR="$(zcat "$R1" | head -n 1 || true)"
    IFS=':' read -r _ _ FC LANE _ _ _ _ _ IDX <<< "$HDR"

    if [[ -z "${FC:-}" || -z "${LANE:-}" ]]; then
        echo "ERRO: cabeçalho inesperado em $R1: $HDR" >&2
        exit 1
    fi

    RG="@RG\tID:${SM}.${FC}.${LANE}\tSM:${SM}\tLB:${LB_PROJETO}\tPU:${FC}.${LANE}.${IDX}\tPL:ILLUMINA\tCN:${CN}\tDT:${DT}"

    echo ">>> $BASE" >&2
    echo "    RG: $RG" >&2

    bwa-mem2 mem \
        -t "$THREADS" \
        -R "$RG" \
        "$GENOME" \
        "$R1" "$R2" \
    | samtools sort \
        -@ "$SORT_THREADS" \
        -T "${OUTDIR}/${BASE}.sorttmp" \
        -o "$TMP" -

    mv "$TMP" "$OUT"
    samtools index -@ "$SORT_THREADS" "$OUT"
done

---

### 5. Marcando duplicatas de PCR

In [ ]:
%%bash

set -e

for bam in *.sorted.bam; do
    base=${bam%.sorted.bam}

    picard MarkDuplicates \
        I="$bam" \
        O="${base}.dedup.bam" \
        M="${base}.metrics.txt" \
        REMOVE_DUPLICATES=false

    samtools index "${base}.dedup.bam"

    rm "$bam"
done

---

### 6. Estatisticas descritivas dos BAM final

In [ ]:
#!/usr/bin/env bash
set -euo pipefail

BAM_DIR="."
OUT="qc_bam"
THREADS=16

mkdir -p "$OUT"/{flagstat,stats,idxstats,mosdepth}

for bam in "$BAM_DIR"/*.bam; do
    sample=$(basename "$bam" .bam)
    echo "[$(date +%H:%M:%S)] Processando $sample"

    # integridade: se falhar, avisa e segue para a próxima amostra
    if ! samtools quickcheck "$bam"; then
        echo "ATENÇÃO: $bam falhou no quickcheck" >&2
        continue
    fi

    [[ -s "$OUT/flagstat/${sample}.flagstat.txt" ]] || \
        samtools flagstat -@ "$THREADS" "$bam" > "$OUT/flagstat/${sample}.flagstat.txt"

    [[ -s "$OUT/stats/${sample}.stats.txt" ]] || \
        samtools stats -@ "$THREADS" "$bam" > "$OUT/stats/${sample}.stats.txt"

    [[ -s "$OUT/idxstats/${sample}.idxstats.txt" ]] || \
        samtools idxstats "$bam" > "$OUT/idxstats/${sample}.idxstats.txt"

    [[ -s "$OUT/mosdepth/${sample}.mosdepth.summary.txt" ]] || \
        mosdepth -t "$THREADS" "$OUT/mosdepth/${sample}" "$bam"
done

multiqc "$OUT" -o multiqc_bam

---

### 7. Mascarando regioes repetitivas Repeat Masker

In [ ]:
%%bash

conda create -n repeatmasker -c conda-forge -c bioconda repeatmasker rmblast trf hmmer

Indexa genoma

In [ ]:
%%bash

samtools faidx genoma.fna

##### 7.1 Executando Repeat Masker

Baixe o RepBase31.10 dentro de: https://www.girinst.org/server/RepBase/protected/RepBase31.10.fasta/ [Selecione o .ref do taxon desejado]  
Consulte: https://girinst.org/repbase/update/
- Utilizamos RepBase de monocotiledôneas `mcotrep.ref`

Execute o script .py para converter o formato .ref no formato FASTA para imput no RepeatMasker

In [ ]:
#!/usr/bin/env python3
"""
Reformata cabeçalhos de um FASTA.

Entrada : >Copia-14_NN-I  Copia   Nelumbo nucifera
Saída   : >Copia-14_NN-I/Copia @Nelumbo_nucifera

Regras:
  - campos separados por 2+ espaços ou tabs (fallback: espaços simples)
  - todos os campos, exceto o último, são unidos por "/"
  - o último campo (espécie) recebe "@" e espaços viram "_"

Uso:
  python reformatar_fasta.py entrada.fasta saida.fasta
"""
import re
import sys


def reformatar_header(linha):
    corpo = linha[1:].strip()
    campos = [c for c in re.split(r"\s{2,}|\t", corpo) if c]

    # fallback: cabeçalho separado por espaços simples
    if len(campos) < 3:
        tokens = corpo.split()
        if len(tokens) < 4:
            return linha.rstrip("\n")  # formato inesperado: mantém original
        campos = [tokens[0], *tokens[1:-2], " ".join(tokens[-2:])]

    *resto, especie = campos
    resto = "/".join(c.replace(" ", "_") for c in resto)
    especie = especie.replace(" ", "_")
    return f">{resto} @{especie}"


def main():
    if len(sys.argv) != 3:
        sys.exit("Uso: python reformatar_fasta.py entrada.fasta saida.fasta")

    with open(sys.argv[1]) as fin, open(sys.argv[2], "w") as fout:
        for linha in fin:
            if linha.startswith(">"):
                fout.write(reformatar_header(linha) + "\n")
            else:
                fout.write(linha)


if __name__ == "__main__":
    main()


> Fluxo de trabalho Bieker et al. 2022 (Uncovering the genomic basis of an extraordinary plant invasion)
- Use o RepeatMasker com uma biblioteca já conhecida ->  
- Com o genoma mascarado utilize o BuildDatabase ->  
- A partir da base construída use o RepeatModeler ->
- A partir da biblioteca personalizada para Macauba ->
- Faça a segunda rodada do RepeatMasker -> 
- `Resultado Final`

Execute o RepeatMask

In [ ]:
%%bash

RepeatMasker \
    -pa 30 \
    -s \
    -lib Monocot_Dfam.fa \
    -xsmall \
    -gff \
    -html \
    macauba_genoma.fna

Execute o RepeatModeler para descoberta _de novo_ de elementos repetitivos a partir do `.masked.` gerado anteriormente

In [ ]:
%%bash

mkdir -p repeatmodeler
cd repeatmodeler

BuildDatabase \
    -name macauba_masked \
    -engine ncbi \
    ../macauba_genoma.fna.masked

Execute o RepeatModeler:

In [ ]:
%%bash

RepeatModeler \
    -database macauba_masked \
    -threads 30

Segunda rodada do RepeatMasker

%%bash

cd ..
mkdir -p repeatmasker_denovo

RepeatMasker \
    -pa 30 \
    -s \
    -lib repeatmodeler/RM_*/consensi.fa.classified \ #atenção !! ajuste o caminho para o RM_* mais recente criado
    -xsmall \
    -gff \
    -html \
    -dir repeatmasker_denovo \
    macauba_genoma.fna.masked

Identifique o `masked.masked`

Converte o output do repeatmasker em BED

Converter Genoma mascarado em um arquivo de coordenadas SEM regiões repetitivas

In [ ]:
#!/usr/bin/env python3

import sys

fasta = sys.argv[1]
output = sys.argv[2]

with open(fasta) as inp, open(output, "w") as out:
    chrom = None
    pos = 0
    start = None

    def close_interval():
        global_dummy = None
        if start is not None:
            out.write(f"{chrom}:{start}-{pos}\n")

    for line in inp:
        line = line.strip()

        if not line:
            continue

        if line.startswith(">"):
            close_interval()
            chrom = line[1:].split()[0]
            pos = 0
            start = None
            continue

        for base in line:
            if base in "ACGT":
                if start is None:
                    start = pos + 1
            else:
                close_interval()
                start = None

            pos += 1

    close_interval()


`Lembre-se de ao rodar o ANGSD utilizar o argumentos -rf (e o arquivo .txt)`

---

### 8. Genotipagem

Conjuntos propostos para teste inicial:

| Conjunto | `-GL` | `-minInd` | `-minQ` | `-minMaf` |
|:--------:|:-----:|:---------:|:-------:|:---------:|
| 1 | 2 | 55 | 20 | 0 |
| 2 | 1 | 55 | 20 | 0 |
| 3 | 2 | 55 | 20 | 0 |
| 4 | 2 | 55 | 30 | 0.05 |

##### 8.1 Estudando argumentos

Entrada e saída

- -b `bam.list`: arquivo com um BAM por linha, as amostras analisadas.
- -ref `ref.fa`: genoma de referência. Precisa ser o mesmo do mapeamento, e é exigido pelo -baq.
- -out `prefixo`: prefixo de todos os arquivos de saída.
- -nThreads `8`: número de threads.

Modelo e frequência alélica

- -GL `2`: modelo de genotype likelihood. 1 é o do samtools e 2 é o do GATK.
- -doMajorMinor `1`: infere o alelo maior e o menor de cada sítio a partir das likelihoods.
- -doMaf `1`: estima a frequência alélica usando esses alelos já definidos. Gera o .mafs.gz.

Filtros de sítio

- -SNP_pval `1e-6`: só mantém sítios cujo teste de razão de verossimilhança indica que são polimórficos com p abaixo do limiar. Quanto menor o valor, mais estrito.
- -minMaf `0`: frequência mínima do alelo menor. 0 não filtra, e 0.05 descarta alelos raros.
- -minInd `20`: número mínimo de indivíduos com dado no sítio. É o filtro de dado faltante.

Filtros de leitura e de base

- -minMapQ `30`: qualidade de mapeamento mínima da leitura.
- -minQ `20`: qualidade mínima da base (Phred).
- -uniqueOnly `1`: descarta leituras que mapeiam em mais de um lugar.
- -remove_bads `1`: remove leituras marcadas como ruins (não mapeadas, secundárias, que falharam no controle de qualidade, duplicatas).
- -only_proper_pairs `1`: mantém só leituras cujo par mapeou de forma consistente.
- -C `50`: reduz a qualidade de mapeamento de leituras com muitos mismatches. 50 é o valor recomendado.
- -baq `1`: recalcula a qualidade das bases perto de indels e alinhamentos duvidosos (BAQ). Reduz falsos SNPs nessas regiões.

Profundidade

- -doCounts `1`: conta as bases em cada sítio, e é pré-requisito do -doDepth.
- -doDepth `1`: gera a distribuição de profundidade por amostra (.depthSample) e global (.depthGlobal).
- -maxDepth `1000`: só define o limite superior do histograma. Não filtra nada.
- -setMinDepth / -setMaxDepth (não estão nos comandos): estes sim filtram sítios pela profundidade somada de todas as amostras.

Saída das likelihoods

- -doGlf `2`: grava as genotype likelihoods em formato Beagle (.beagle.gz), o que o PCAngsd lê.

Flags do PCAngsd

- -doIBS 1 -makeMatrix 1 -doCov 1, que geram `.ibsMat` e `.covMat`

> Teste 1

In [ ]:
%%bash

angsd -b bam.list -ref ref.fa -out set1_base_GL2_ind55 -rf regioes.txt \
  -GL 2 -doMajorMinor 1 -doMaf 1 -SNP_pval 1e-6 -minMaf 0 \
  -minMapQ 30 -minQ 20 -minInd 55 \
  -doIBS 1 -makeMatrix 1 -doCov 1 \
  -uniqueOnly 1 -remove_bads 1 -only_proper_pairs 1 -C 50 -baq 1 \
  -doCounts 1 -doDepth 1 -maxDepth 1000 -doGlf 2 \
  -nThreads 8

> Teste 2

In [ ]:
%%bash

angsd -b bam.list -ref ref.fa -out set2_GL1_ind55 -rf regioes.txt \
  -GL 1 -doMajorMinor 1 -doMaf 1 -SNP_pval 1e-6 -minMaf 0 \
  -minMapQ 30 -minQ 20 -minInd 55 \
  -doIBS 1 -makeMatrix 1 -doCov 1 \
  -uniqueOnly 1 -remove_bads 1 -only_proper_pairs 1 -C 50 -baq 1 \
  -doCounts 1 -doDepth 1 -maxDepth 1000 -doGlf 2 \
  -nThreads 8

> Teste 3

In [ ]:
%%bash

angsd -b bam.list -ref ref.fa -out set3_GL2_ind55 -rf regioes.txt \
  -GL 2 -doMajorMinor 1 -doMaf 1 -SNP_pval 1e-6 -minMaf 0 \
  -minMapQ 30 -minQ 20 -minInd 55 \
  -doIBS 1 -makeMatrix 1 -doCov 1 \
  -uniqueOnly 1 -remove_bads 1 -only_proper_pairs 1 -C 50 -baq 1 \
  -doCounts 1 -doDepth 1 -maxDepth 1000 -doGlf 2 \
  -nThreads 8

> Teste 4

In [ ]:
%%bash

angsd -b bam.list -ref ref.fa -out set4_GL2_Q30_maf55 -rf regioes.txt \
  -GL 2 -doMajorMinor 1 -doMaf 1 -SNP_pval 1e-6 -minMaf 0.05 \
  -minMapQ 30 -minQ 30 -minInd 55 \
  -doIBS 1 -makeMatrix 1 -doCov 1 \
  -uniqueOnly 1 -remove_bads 1 -only_proper_pairs 1 -C 50 -baq 1 \
  -doCounts 1 -doDepth 1 -maxDepth 1000 -doGlf 2 \
  -nThreads 8

Outputs:

| `Arquivo` |	`Origem` |	`Visualização` |
|---------|--------|-----------------|
| .arg |	parâmetros efetivos |	impresso para registro |
| .depthGlobal |	-doDepth |	distribuição de profundidade global |
| .depthSample |	-doDepth |	profundidade média e proporção de sítios cobertos por indivíduo, com tabela interativa |
| .mafs.gz |	-doMaf / -SNP_pval |	espectro de frequências, MAF dobrada, nInd com a linha do -minInd, valor-p do SNP, Ti/Tv e densidade de SNPs por janela e por sequência |
| .beagle.gz |	-doGlf 2 |	completude e certeza genotípica por indivíduo |
| .ibsMat	| -doIBS + -makeMatrix |	heatmap, árvore NJ e PCoA |
| .covMat |	-doCov |	scree plot, PC1 a PC3 e heatmap |
| .pos.gz | -doCounts | habilita contagem de bases A, C, T, G |